In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from benchmark_data_leakage.utils import find_repo_root

repo_root = find_repo_root()
fig_dir = repo_root / "figures"
fig_dir.mkdir(exist_ok=True)

In [ ]:
NULL_X = 1.05

# For reproducing, use:
# https://github.com/bamattsson/nfab/tree/56a1ab122ca9c36233089ba3f81f0bebb7f96ce8
# and follow README.md (1->2) and docs/baseline.md (1->3) with the following changes
# benchmark.yaml:
#   pipeline.tanimoto_threshold: (vary this 0.3 -> 1.0 and then null)
# train.yaml:
#   model.use_*: true/false
# Prediction command (step 3) should be with --n-bootstraps 1000

# threshold_numeric, model, pearson_r, SE, n_rows, n_assays
raw_data = [
    # FP + Mol Prop
    (0.30,  "FP + Mol Prop", -0.0527, 0.1119,   603,   6),
    (0.35,  "FP + Mol Prop",  0.0150, 0.0601,  1108,  24),
    (0.40,  "FP + Mol Prop",  0.1572, 0.0357,  2680,  81),
    (0.50,  "FP + Mol Prop",  0.1533, 0.0212,  7593, 255),
    (0.70,  "FP + Mol Prop",  0.2044, 0.0142, 17336, 591),
    (1.0,   "FP + Mol Prop",  0.2021, 0.0117, 21025, 703),
    (NULL_X,"FP + Mol Prop",  0.3086, 0.0116, 28568, 910),
    # FP only
    (0.30,  "FP only",  0.1261, 0.0508,   603,   6),
    (0.35,  "FP only",  0.1238, 0.0710,  1108,  24),
    (0.40,  "FP only",  0.2102, 0.0350,  2680,  81),
    (0.50,  "FP only",  0.1902, 0.0191,  7593, 255),
    (0.70,  "FP only",  0.2007, 0.0133, 17336, 591),
    (1.0,   "FP only",  0.2167, 0.0119, 21025, 703),
    (NULL_X,"FP only",  0.3164, 0.0116, 28568, 910),
]

df = pd.DataFrame(raw_data, columns=["threshold_numeric", "model", "pearson_r", "SE", "n_rows", "n_assays"])
df

In [ ]:
C_FP_MOL   = "#93cfe0"   # pale teal   – FP + Mol Prop
C_FP_ONLY  = "#e8a8b0"   # pale rose   – FP only

model_colors = {
    "FP + Mol Prop": C_FP_MOL,
    "FP only":       C_FP_ONLY,
}

models_to_plot = ["FP + Mol Prop", "FP only"]

# Sorted threshold values; null sits at 1.2 on the numeric axis
real_thresholds = sorted(df.loc[df["threshold_numeric"] < NULL_X, "threshold_numeric"].unique())
thresholds_in_order = real_thresholds + [NULL_X]

# Bar geometry: proportional x-axis using actual threshold values.
# Bars are offset symmetrically around each threshold tick.
n_models  = len(models_to_plot)
bar_width = 0.016
bar_step  = bar_width * 1.25   # centre-to-centre within a group
half_span = bar_step * (n_models - 1) / 2
offsets   = np.linspace(-half_span, half_span, n_models)

fig, ax = plt.subplots(figsize=(9.0, 4.8))

for m_idx, model in enumerate(models_to_plot):
    mdf = df[df["model"] == model].set_index("threshold_numeric")
    xs, ys, ses = [], [], []
    for t in thresholds_in_order:
        xs.append(t + offsets[m_idx])
        ys.append(mdf.loc[t, "pearson_r"])
        ses.append(mdf.loc[t, "SE"])

    bars = ax.bar(
        xs, ys,
        width=bar_width,
        color=model_colors[model],
        edgecolor="#323232", linewidth=0.6,
        yerr=ses, capsize=2,
        error_kw=dict(elinewidth=0.8, ecolor="#323232", capthick=0.8, zorder=5),
        zorder=2,
        label=model,
    )

    for bar, val, se in zip(bars, ys, ses):
        x = bar.get_x() + bar.get_width() / 2
        y_txt = val + se + 0.025   # always above the top of the error bar
        ax.text(x, y_txt, f"{val:.2f}", ha="center", va="bottom",
                fontsize=7, color="black", rotation=90)

# X-axis: numeric ticks at threshold positions; null labelled "no filter"
tick_labels = [str(t) if t != NULL_X else "no filter" for t in thresholds_in_order]
ax.set_xticks(thresholds_in_order)
ax.set_xticklabels(tick_labels, fontsize=9)
ax.set_xlabel("Tanimoto threshold", fontsize=11)
ax.set_ylabel("Mean Pearson correlation ($r$)", fontsize=11)

# Pad x limits so edge bars aren't clipped
ax.set_xlim(min(thresholds_in_order) - 0.04, max(thresholds_in_order) + 0.04)
ax.set_ylim(-0.2, 0.60)

ax.grid(visible=True, which="major", axis="y", linestyle=":", linewidth=0.5, color="#cccccc")
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", length=0, labelsize=9)
ax.tick_params(axis="y", labelsize=9)

legend_handles = [
    mpatches.Patch(color=model_colors[m], edgecolor="#323232", linewidth=0.6, label=m)
    for m in models_to_plot
]
ax.legend(handles=legend_handles, fontsize=9, frameon=False, loc="upper left")

plt.tight_layout()
plt.savefig(fig_dir / "tanimoto_threshold_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
table = (
    df.drop_duplicates("threshold_numeric")
    .set_index("threshold_numeric")
    .loc[thresholds_in_order, ["n_rows", "n_assays"]]
    .rename(columns={"n_rows": "n_measurements"})
    .rename(index={NULL_X: "no filter"})
)
table.index.name = "threshold"
table